# TFT (Temporal Fusion Transformer) Simulation — Shredder Bearing Temperature Prediction

## 개요

이 노트북은 **TFT(Temporal Fusion Transformer)**의 핵심 원리를 체험하는 시뮬레이션입니다.

| 항목 | 내용 |
|------|------|
| **모델** | TFT (Google, 2019) |
| **시나리오** | 슈레더 베어링 온도 90일 데이터로 24시간 후 예측 |
| **핵심 원리** | 3가지 입력 유형 + Attention + Variable Selection |

### TFT의 3가지 입력 유형

```
TFT Input Types:

  1. Time-varying Unknown  → 과거 센서 데이터 (미래 값을 모름)
     온도, 진동, 전류, RPM, 처리량

  2. Time-varying Known    → 미래에도 알 수 있는 정보
     시간(hour), 요일(day_of_week), 주말 여부

  3. Static Covariates     → 시간에 따라 변하지 않는 정보
     장비 ID, 칼날 교체 후 경과시간
```

### Prophet / LSTM과의 비교

| 비교 | Prophet | LSTM | **TFT** |
|:---:|:---:|:---:|:---:|
| 입력 변수 | 1개 (단변량) | 여러 개 (다변량) | **3유형 동시 (다변량+시간+정적)** |
| 입력 구분 | 없음 | 없음 | **3유형 구분 처리** |
| 변수 중요도 | 없음 | 없음 | **Variable Selection** |
| Attention | 없음 | 없음 | **Multi-head Attention** |
| 해석 가능성 | 높음 (분해) | 낮음 (블랙박스) | **높음 (Attention + 중요도)** |

> **참고**: 실제 TFT는 `pytorch-forecasting` 라이브러리가 필요하고 매우 무겁습니다.  
> 이 시뮬레이션은 `GradientBoostingRegressor`로 TFT의 **다입력 + 변수 중요도 + Attention** 개념을 체험합니다.

---
## Step 0. 라이브러리 설치 및 Import

In [ ]:
!pip install -q scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('All libraries loaded successfully!')

---
## Step 1. Shredder Sensor Data Generation

산업용 슈레더(Shredder)의 센서 데이터를 시뮬레이션합니다.

### 생성되는 센서 데이터

| 센서 | 단위 | 정상 범위 | 설명 |
|------|------|-----------|------|
| temperature | °C | 20~38 | 베어링 온도 (예측 대상) |
| vibration | mm/s | 1.5~4.0 | 진동 RMS |
| current | A | 65~110 | 모터 전류 |
| rpm | rpm | 19~21 | 회전 속도 |
| throughput | t/h | 1.5~3.5 | 처리량 |

### TFT는 5개 센서를 **모두 동시에** 사용합니다!

- **Prophet**: 온도 1개만 사용 (단변량)
- **LSTM**: 온도+진동+전류 3개 사용
- **TFT**: 5개 센서 + 시간 정보 + 정적 정보 **모두** 사용

In [ ]:
def generate_shredder_data(days=90, freq_minutes=10, seed=42):
    """
    Shredder sensor data simulator.
    Generates realistic bearing temperature, vibration, current, rpm, throughput.
    """
    np.random.seed(seed)

    n_points = days * 24 * 60 // freq_minutes
    timestamps = pd.date_range(
        start='2026-01-01',
        periods=n_points,
        freq=f'{freq_minutes}min'
    )

    t = np.arange(n_points)
    hours = np.array([ts.hour for ts in timestamps])
    dow = np.array([ts.dayofweek for ts in timestamps])

    # === Bearing Temperature ===
    base_temp = 28.0
    daily_pattern = 5.0 * np.sin(2 * np.pi * hours / 24 - np.pi/2)
    weekly_pattern = np.where(dow >= 5, -3.0, 0.0)
    wear_trend = 0.03 * t / (24 * 60 / freq_minutes)
    noise = np.random.normal(0, 0.8, n_points)

    anomaly_mask = np.random.random(n_points) < 0.005
    anomaly_spike = anomaly_mask * np.random.uniform(15, 30, n_points)

    temperature = base_temp + daily_pattern + weekly_pattern + wear_trend + noise + anomaly_spike
    temperature = np.clip(temperature, 15, 80)

    # === Vibration RMS ===
    base_vib = 2.5
    vib_daily = 0.5 * np.sin(2 * np.pi * hours / 24)
    vib_wear = 0.02 * t / (24 * 60 / freq_minutes)
    vib_noise = np.random.normal(0, 0.3, n_points)
    vib_anomaly = anomaly_mask * np.random.uniform(5, 15, n_points)

    vibration = base_vib + vib_daily + vib_wear + vib_noise + vib_anomaly
    vibration = np.clip(vibration, 0.5, 25)

    # === Motor Current ===
    base_cur = 85.0
    cur_daily = 10.0 * np.sin(2 * np.pi * hours / 24 - np.pi/3)
    cur_wear = 0.05 * t / (24 * 60 / freq_minutes)
    cur_noise = np.random.normal(0, 2.0, n_points)
    cur_weekend = np.where(dow >= 5, -30.0, 0.0)

    current = base_cur + cur_daily + cur_wear + cur_noise + cur_weekend
    current = np.clip(current, 20, 150)

    # === RPM ===
    base_rpm = 20.0
    rpm_var = np.random.normal(0, 0.3, n_points)
    rpm_weekend = np.where(dow >= 5, -15.0, 0.0)

    rpm = base_rpm + rpm_var + rpm_weekend
    rpm = np.clip(rpm, 0, 25)

    # === Throughput ===
    base_tp = 2.5
    tp_daily = 0.5 * np.sin(2 * np.pi * hours / 24 - np.pi/4)
    tp_noise = np.random.normal(0, 0.15, n_points)
    tp_weekend = np.where(dow >= 5, -2.0, 0.0)

    throughput = base_tp + tp_daily + tp_noise + tp_weekend
    throughput = np.clip(throughput, 0, 4)

    df = pd.DataFrame({
        'timestamp': timestamps,
        'temperature': np.round(temperature, 2),
        'vibration': np.round(vibration, 2),
        'current': np.round(current, 2),
        'rpm': np.round(rpm, 2),
        'throughput': np.round(throughput, 2)
    })

    return df

print('generate_shredder_data() defined.')

In [ ]:
# Generate 90 days of data, sampled every 1 hour
df = generate_shredder_data(days=90, freq_minutes=60)

print(f'Generated data: {len(df)} samples')
print(f'Period: {df["timestamp"].min()} ~ {df["timestamp"].max()}')
print(f'\nTFT uses ALL 5 sensors simultaneously!')
print(f'\nStatistics:')
df.describe().round(2)

---
## Step 2. Raw Data Visualization

생성된 원본 데이터를 확인합니다. TFT는 5개 센서 **모두**를 동시에 사용합니다.

**관찰 포인트:**
- 온도(temperature)에 일간/주간 패턴이 보이는가?
- 90일간 온도가 서서히 상승하는 트렌드가 보이는가?
- 진동, 전류, RPM, 처리량과 온도 간 상관관계가 보이는가?
- TFT는 이 모든 센서를 함께 활용하여 예측합니다

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True)

sensors = [
    ('temperature', 'Bearing Temperature', 'C', 'tab:red'),
    ('vibration', 'Vibration RMS', 'mm/s', 'tab:blue'),
    ('current', 'Motor Current', 'A', 'tab:green'),
    ('rpm', 'RPM', 'rpm', 'tab:orange'),
    ('throughput', 'Throughput', 't/h', 'tab:purple'),
]

for ax, (col, title, unit, color) in zip(axes, sensors):
    ax.plot(df['timestamp'], df[col], color=color, alpha=0.7, linewidth=0.5)
    ax.set_ylabel(f'{title} ({unit})', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_title(f'{title}', fontsize=12, fontweight='bold')

axes[-1].set_xlabel('Date', fontsize=11)
fig.suptitle('Shredder Sensor Data — 90 Days Overview (TFT uses ALL 5 sensors)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## Step 3. TFT의 3가지 입력 유형 (Feature Engineering)

TFT의 핵심 차별점은 입력을 **3가지 유형으로 구분**하여 처리하는 것입니다.

| 입력 유형 | 설명 | 예시 | 다른 모델 |
|-----------|------|------|----------|
| **Time-varying Unknown** | 과거에만 있고 미래를 모르는 정보 | 센서 값, lag, rolling 통계 | Prophet/LSTM도 사용 가능 |
| **Time-varying Known** | 미래에도 알 수 있는 정보 | hour, day_of_week, is_weekend | ★ TFT만 구분 처리 |
| **Static Covariates** | 시간에 따라 변하지 않는 정보 | blade_age, equipment_id | ★ TFT만 구분 처리 |

### 왜 이 구분이 중요한가?

- **Prophet/LSTM**: 모든 입력을 동일하게 취급 → "미래에도 아는 정보"를 특별히 활용 못함
- **TFT**: "내일은 월요일이다"라는 미래 정보를 **알면서 활용** → 예측 정확도 향상

In [ ]:
# === TFT Feature Engineering ===

features = pd.DataFrame()

# ──────────────────────────────────────────
# 1. Time-varying Unknown (past sensor data)
# ──────────────────────────────────────────
for col in ['temperature', 'vibration', 'current', 'rpm', 'throughput']:
    features[col] = df[col].values
    # Lag features (past values)
    for lag in [1, 3, 6, 12, 24]:
        features[f'{col}_lag{lag}'] = df[col].shift(lag).values
    # Rolling statistics
    features[f'{col}_roll_mean_12'] = df[col].rolling(12).mean().values
    features[f'{col}_roll_std_12'] = df[col].rolling(12).std().values

# ──────────────────────────────────────────
# 2. Time-varying Known (future-known info)
# ──────────────────────────────────────────
features['hour'] = df['timestamp'].dt.hour
features['day_of_week'] = df['timestamp'].dt.dayofweek
features['is_weekend'] = (df['timestamp'].dt.dayofweek >= 5).astype(int)
features['hour_sin'] = np.sin(2 * np.pi * features['hour'] / 24)
features['hour_cos'] = np.cos(2 * np.pi * features['hour'] / 24)

# ──────────────────────────────────────────
# 3. Static Covariates (time-invariant info)
# ──────────────────────────────────────────
features['blade_age_hours'] = np.arange(len(df))  # hours since blade replacement
features['equipment_id'] = 0  # single shredder

# ──────────────────────────────────────────
# Target: temperature 24 hours ahead
# ──────────────────────────────────────────
target = df['temperature'].shift(-24)

# Remove NaN rows
valid = features.notna().all(axis=1) & target.notna()
features = features[valid].reset_index(drop=True)
target = target[valid].reset_index(drop=True)
df_valid = df[valid].reset_index(drop=True)

feature_names = features.columns.tolist()

# Count features per group
n_unknown = sum(1 for f in feature_names if any(c in f for c in ['temperature','vibration','current','rpm','throughput']) and 'hour' not in f and 'day' not in f and 'weekend' not in f and 'blade' not in f and 'equip' not in f)
n_known = sum(1 for f in feature_names if any(c in f for c in ['hour', 'day_of_week', 'is_weekend']))
n_static = sum(1 for f in feature_names if any(c in f for c in ['blade', 'equip']))

print(f'Total features: {len(feature_names)}')
print(f'  Time-varying Unknown (sensors + lags + rolling): {n_unknown}')
print(f'  Time-varying Known (hour, day, weekend, sin/cos): {n_known}')
print(f'  Static (blade_age, equipment_id): {n_static}')
print(f'\nTarget: temperature shifted by -24 (predict 24h ahead)')
print(f'Valid samples: {len(features)}')

### TFT 3가지 입력 유형 정리

| 유형 | 특징 이름 | 개수 | 설명 |
|------|-----------|------|------|
| **Time-varying Unknown** | 센서값, lag1~24, rolling mean/std | ~35개 | 과거 센서 데이터 (미래 값 모름) |
| **Time-varying Known** | hour, day_of_week, is_weekend, hour_sin/cos | 5개 | 미래에도 알 수 있는 시간 정보 |
| **Static** | blade_age_hours, equipment_id | 2개 | 시간 불변 정보 |

→ Prophet은 **단변량만**, LSTM은 **구분 없이 다변량**, TFT는 **3유형 구분 처리!**

---
## Step 4. Train/Test Split

시계열 데이터는 **반드시 시간순으로 분할**해야 합니다.

```
|<--- Train (70%) --->|<--- Test (30%) --->|
|     Jan ~ early Mar  |   mid Mar ~ end Mar |
```

In [ ]:
# Time-ordered split (no random shuffle!)
train_size = int(len(features) * 0.7)

X_train = features.iloc[:train_size]
y_train = target.iloc[:train_size]
X_test = features.iloc[train_size:]
y_test = target.iloc[train_size:]

print(f'Train: {len(X_train)} samples ({df_valid["timestamp"].iloc[0].date()} ~ {df_valid["timestamp"].iloc[train_size-1].date()})')
print(f'Test : {len(X_test)} samples ({df_valid["timestamp"].iloc[train_size].date()} ~ {df_valid["timestamp"].iloc[-1].date()})')
print(f'\nFeatures per sample: {X_train.shape[1]}')

In [ ]:
# Visualize train/test split
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_valid['timestamp'].iloc[:train_size], y_train.values,
        'b.', alpha=0.3, markersize=2, label='Train')
ax.plot(df_valid['timestamp'].iloc[train_size:], y_test.values,
        'orange', alpha=0.5, markersize=2, label='Test', marker='.', linestyle='none')
ax.axvline(x=df_valid['timestamp'].iloc[train_size], color='green',
           linestyle='--', linewidth=2, label='Split boundary')
ax.set_title('Train / Test Split (70% / 30%) — Target: Temperature 24h Ahead',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Temperature (C)')
ax.set_xlabel('Date')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 5. Model Training (TFT Concept Simulator)

실제 TFT는 Transformer 구조 기반의 매우 무거운 모델입니다 (`pytorch-forecasting` 필요).  
이 시뮬레이션에서는 `GradientBoostingRegressor`를 사용하여 TFT의 **핵심 개념**을 체험합니다:

| TFT 기능 | GBM 시뮬레이션 |
|----------|---------------|
| 3가지 입력 유형 | 동일하게 3유형 특징 구성 후 입력 |
| Variable Selection | `feature_importances_`로 변수 중요도 확인 |
| Multi-head Attention | lag 특징의 중요도로 시간 attention 시뮬레이션 |
| 24시간 예측 | target = temperature.shift(-24) |

**GBM 하이퍼파라미터:**
- `n_estimators=300`: 트리 300개 앙상블
- `max_depth=8`: 복잡한 다변량 관계 포착
- `learning_rate=0.05`: 점진적 학습

In [ ]:
print('Training TFT concept simulator (GradientBoostingRegressor)...')
print(f'  n_estimators=300, max_depth=8, learning_rate=0.05')

model = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print('\nTraining complete!')
print(f'  Input features: {X_train.shape[1]}')
print(f'  Training samples: {len(X_train)}')

---
## Step 6. Performance Evaluation

24시간 후 온도 예측의 정확도를 평가합니다.

| 지표 | 의미 |
|------|------|
| **MAE** | 평균 절대 오차 — 평균적으로 몇 도 틀리는가 |
| **RMSE** | 평균 제곱근 오차 — 큰 오차에 더 민감 |
| **R²** | 결정계수 — 1에 가까울수록 좋음 |

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print('=' * 50)
print('  TFT Simulation — Performance Evaluation')
print('=' * 50)
print(f'  MAE  = {mae:.2f} C (24시간 후 온도 예측 평균 오차)')
print(f'  RMSE = {rmse:.2f} C')
print(f'  R2   = {r2:.4f}')
print('=' * 50)

---
## Step 7. Result Visualization

### 7-1. 24h Ahead Forecast vs Actual

테스트 구간의 처음 200포인트에서 실제 온도(24시간 후)와 TFT 예측을 비교합니다.

In [ ]:
test_timestamps = df_valid['timestamp'].iloc[train_size:train_size + len(y_test)]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_timestamps.values[:200], y_test.values[:200], 'b-', alpha=0.7, linewidth=1.5, label='Actual (24h ahead)')
ax.plot(test_timestamps.values[:200], y_pred[:200], 'r--', alpha=0.8, linewidth=1.5, label='TFT Forecast')
ax.set_title(f'24h Ahead Temperature Forecast (MAE={mae:.2f} C, R2={r2:.4f})',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Temperature (C)')
ax.set_xlabel('Date')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7-2. Variable Group Importance (TFT Variable Selection 시뮬레이션)

TFT의 핵심 기능: **어떤 센서 그룹이 예측에 가장 중요한지** 자동으로 파악합니다.  
이것은 Prophet(해석 불가)이나 LSTM(블랙박스)에서는 제공하지 않는 기능입니다.

In [ ]:
# Calculate feature importances and group by sensor type
importance = pd.Series(model.feature_importances_, index=feature_names)
importance = importance.sort_values(ascending=False)

# Group importance by category
group_imp = {}
for name, imp in importance.items():
    if 'temperature' in name:
        group_imp['Temperature'] = group_imp.get('Temperature', 0) + imp
    elif 'vibration' in name:
        group_imp['Vibration'] = group_imp.get('Vibration', 0) + imp
    elif 'current' in name:
        group_imp['Current'] = group_imp.get('Current', 0) + imp
    elif any(s in name for s in ['hour', 'day', 'weekend']):
        group_imp['Time Info'] = group_imp.get('Time Info', 0) + imp
    elif any(s in name for s in ['blade', 'equip']):
        group_imp['Static Info'] = group_imp.get('Static Info', 0) + imp
    else:
        group_imp['Other'] = group_imp.get('Other', 0) + imp

sorted_groups = sorted(group_imp.items(), key=lambda x: -x[1])

# Pie chart
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#dc2626', '#2563eb', '#059669', '#d97706', '#7c3aed', '#94a3b8']
ax.pie([v for _, v in sorted_groups],
       labels=[k for k, _ in sorted_groups],
       colors=colors[:len(sorted_groups)],
       autopct='%1.1f%%', startangle=90,
       textprops={'fontsize': 11})
ax.set_title('TFT Variable Selection Simulation\n"Which sensor group matters most?"',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Variable group importance:')
for group, imp in sorted_groups:
    bar = '=' * int(imp * 50)
    print(f'  {group:15s}: {imp:.4f} ({imp*100:.1f}%) {bar}')

### 7-3. Temporal Attention Simulation

TFT의 **Multi-head Attention** 시뮬레이션: "과거의 어떤 시점이 미래 예측에 가장 중요한가?"

- LSTM은 순서대로만 볼 수 있음 (Sequential)
- TFT는 중요한 과거 시점을 **직접 참조** (Attention) → 더 정확한 예측

In [ ]:
# Aggregate importance by lag distance
lag_importance = {}
for name, imp in importance.items():
    if '_lag' in name:
        lag = int(name.split('_lag')[1])
        lag_label = f'{lag}h ago'
        lag_importance[lag_label] = lag_importance.get(lag_label, 0) + imp

# Sort by lag value for display
lag_sorted = sorted(lag_importance.items(), key=lambda x: -x[1])

fig, ax = plt.subplots(figsize=(10, 5))
labels = [k for k, _ in lag_sorted]
values = [v for _, v in lag_sorted]
bar_colors = ['#dc2626' if v == max(values) else '#2563eb' for v in values]
ax.barh(labels, values, color=bar_colors, alpha=0.8, edgecolor='white')
ax.invert_yaxis()
ax.set_title('TFT Attention Simulation\n"Which past time points matter most for 24h forecast?"',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Importance (sum of all sensors at this lag)')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print('Temporal attention (lag importance):')
for lag, imp in lag_sorted:
    bar = '=' * int(imp * 200)
    print(f'  {lag:10s}: {imp:.4f} {bar}')

### 7-4. Model Comparison

시계열 예측 모델들의 상대적 성능을 비교합니다 (다변량 장기 예측 시나리오 기준).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

models = ['ARIMA\n(Statistics)', 'Prophet\n(Meta)', 'XGBoost\n(Tree)', 'LSTM\n(Deep Learning)', 'TFT\n(Transformer)']
scores = [2, 3, 4, 4.5, 5]  # relative conceptual scores
colors_bar = ['#94a3b8', '#fbbf24', '#2563eb', '#7c3aed', '#059669']

bars = ax.bar(models, scores, color=colors_bar, alpha=0.8, edgecolor='white', linewidth=2)
ax.set_title('Model Comparison — Relative Performance\n(Multi-variate Long-horizon Forecasting)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Relative Score')
ax.set_ylim(0, 6)

for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
            f'{score}', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### 7-5. Error Distribution

예측 오차(실제 - 예측)의 분포를 확인합니다.  
정규분포에 가까우면 모델이 체계적 편향 없이 잘 작동하고 있다는 의미입니다.

In [ ]:
errors = y_test.values - y_pred

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(errors, bins=50, color='steelblue', alpha=0.7, edgecolor='white', density=True)
ax.axvline(x=0, color='red', linewidth=2, linestyle='--', label='Zero error')
ax.axvline(x=np.mean(errors), color='orange', linewidth=2, label=f'Mean error: {np.mean(errors):.2f} C')

ax.set_title('Forecast Error Distribution (Actual - Predicted)', fontsize=13, fontweight='bold')
ax.set_xlabel('Error (C)')
ax.set_ylabel('Density')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Error statistics:')
print(f'  Mean  : {np.mean(errors):+.2f} C')
print(f'  Std   : {np.std(errors):.2f} C')
print(f'  Median: {np.median(errors):+.2f} C')
print(f'  Within +/- 2 C: {(np.abs(errors) < 2).mean()*100:.1f}%')

### 7-6. Test Period Close-up (First 5 Days)

테스트 구간의 처음 5일을 확대하여 24시간 후 예측의 정밀도를 확인합니다.

In [ ]:
n_closeup = 5 * 24  # 5 days

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_timestamps.values[:n_closeup], y_test.values[:n_closeup],
        'b-o', markersize=3, linewidth=1, alpha=0.7, label='Actual (24h ahead)')
ax.plot(test_timestamps.values[:n_closeup], y_pred[:n_closeup],
        'r-s', markersize=3, linewidth=1, alpha=0.7, label='TFT Forecast')

ax.set_title('Test Period Close-up (First 5 Days) — 24h Ahead Forecast',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Temperature (C)')
ax.set_xlabel('Date')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 8. Summary

### TFT의 3가지 입력 유형과 슈레더 적용

| 입력 유형 | TFT에서의 역할 | 슈레더 현장 의미 |
|-----------|---------------|------------------|
| **Time-varying Unknown** | 과거 센서 패턴 학습 | 온도/진동/전류/RPM/처리량 (미래 모름) |
| **Time-varying Known** | 미래 시간 정보 활용 | "내일 월요일이면 가동 시작" (미래도 앎) |
| **Static** | 장비 특성 반영 | 칼날 교체 이력, 장비 ID (변하지 않음) |

### 장점
1. **3가지 입력 유형 동시 처리** — Prophet(단변량), LSTM(구분 없음)과 차별화
2. **Variable Selection** — "어떤 센서가 중요한지" 자동 파악 + 시각화
3. **Multi-head Attention** — "과거 어떤 시점이 중요한지" 직접 참조
4. **Quantile Output** — 불확실성까지 제공 ("90% 신뢰 구간")

### 단점
1. **데이터 많이 필요** — 수천~수만 건 이상 필요 (소규모 데이터에 비효율)
2. **모델이 무거움** — Edge 배포 어려움 (서버 기반 추론 필요)
3. **학습 시간이 김** — GPU 필요, 튜닝 복잡
4. **pytorch-forecasting 필요** — 라이브러리 의존성 높음

### 슈레더 적용 권장
- **적합**: 서버 기반 통합 장기 예측 (24h+), 다변량 예측, 변수 중요도 분석
- **부적합**: Edge 실시간 추론 (→ XGBoost/LSTM 사용), 소규모 데이터

---

> **모델 선택 가이드**:  
> - 단변량 트렌드 분석 → `01_Prophet`  
> - 다변량 이상 탐지 → `02_LSTM`  
> - 다변량 장기 예측 + 해석 → `03_TFT` (이 노트북)  
> - 빠른 다변량 예측 → `04_XGBoost`  
> - 기초 통계 예측 → `05_ARIMA`

In [ ]:
print('=' * 60)
print('  TFT Simulation Complete!')
print('=' * 60)
print(f'''
  Model: TFT (Temporal Fusion Transformer) Concept Simulator
  Simulator: GradientBoostingRegressor (300 trees, depth=8)

  Input Types (TFT's 3 types):
    1. Time-varying Unknown: {n_unknown} features (sensors + lags + rolling)
    2. Time-varying Known  : {n_known} features (hour, day, weekend, sin/cos)
    3. Static Covariates   : {n_static} features (blade_age, equipment_id)
    Total: {len(feature_names)} features

  Target: Temperature 24 hours ahead

  Performance:
    MAE  = {mae:.2f} C
    RMSE = {rmse:.2f} C
    R2   = {r2:.4f}

  Variable Selection (top group): {sorted_groups[0][0]} ({sorted_groups[0][1]*100:.1f}%)

  TFT vs Others:
    Prophet: 단변량만, 트렌드+계절성 분해
    LSTM   : 다변량 가능, but 입력 구분 없음, 블랙박스
    TFT    : 3유형 구분 + Variable Selection + Attention + 해석 가능!
''')